# Patent Applicant Name Harmonization — Stage 1: Generic Cross-Sector Backbone

This notebook implements a **domain-agnostic, unsupervised entity-resolution pipeline** designed to harmonize raw PATSTAT patent applicant strings (`psn_name`) into initial canonical corporate, academic, and public research entities (`harm_id`, `harmonized_name`) without labeled training data.

### Architecture Overview
1. **EPO / PATSTAT Sector Imputation & Filtering:** Replaces `UNKNOWN` or missing `psn_sector` tags using multilingual corporate, academic, government/non-profit, and personal-name rules so legitimate organizations labeled `UNKNOWN` are retained while individual inventors are excluded.
2. **Deterministic Text Normalization:** Standardizes Unicode/diacritics, extracts Latin equivalents appended to CJK strings, expands/abbreviates recurring institutional terms (`UNIVERSITY` $\rightarrow$ `UNIV`, `TECHNOLOGY` $\rightarrow$ `TECH`), and strips legal entity suffixes (`INC`, `LTD`, `GMBH`, `SPA`, CJK corporate forms).
3. **Multi-Signal Pairwise Similarity:** Combines three complementary signals:
   * **Multilingual Semantic Cosine Similarity** (`paraphrase-multilingual-MiniLM-L12-v2`)
   * **Corpus IDF-Weighted Token Jaccard**
   * **Character Edit Ratio** (`RapidFuzzy`, with a zero-token-overlap typo guard `ed >= 0.80`)
4. **Unsupervised Thresholding & Graph Clustering:** Automatically determines the similarity cutoff via **Otsu bimodal thresholding** (bounded below by `MIN_OTSU_FLOOR = 0.56`) and partitions the similarity graph via **Louvain community detection** (`resolution = 1.5`).

*Downstream handoff:* Exports `outputs/harmonization_lookup_stage1.csv` and `outputs/harmonized_output_stage1.csv` for consumption by the domain-specific rectification notebook (`stage2_3_specific_rectification.ipynb`).

## 1. Imports & Pipeline Configuration

Configures relative paths (`../qc_applicants_PROD.csv` as input and `outputs/` for generated artifacts), embedding model settings, similarity weights, and Louvain clustering hyperparameters.

In [3]:
%pip install rapidfuzz sentence_transformers


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [12]:

import os
import re
import unicodedata
from collections import Counter
import numpy as np
import pandas as pd
import networkx as nx
import networkx.algorithms.community as nx_comm
from rapidfuzz.fuzz import ratio as rf_ratio
from sentence_transformers import SentenceTransformer

# --- Paths & Folder Setup ---
INPUT_FOLDER = ".."  # parent directory containing qc_applicants_PROD.csv

INPUT_CSV = os.path.join(INPUT_FOLDER, "PATSTAT_500_label_datav4.csv")

SECTOR_COL = "psn_sector"

NAME_COL = "comp"
WEIGHT_COL = None  # optional: e.g. "n_families" to weight canonical name selection

# --- Stage 1: Generic Multi-Signal Graph Parameters ---
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
SCORE_WEIGHTS = {"cosine": 0.34, "jaccard": 0.33, "edit": 0.33}
SCORE_FLOOR = 0.45
MIN_OTSU_FLOOR = 0.56
LOUVAIN_RESOLUTION_S1 = 1.5
LOUVAIN_SEED = 42

## 2. EPO/PATSTAT Sector Imputation & Deterministic Text Normalization

1. **EPO/PATSTAT Sector Imputation (`impute_psn_sector`):** Preserves known PATSTAT `psn_sector` tags (`COMPANY`, `UNIVERSITY`, `GOV NON-PROFIT`, `INDIVIDUAL`) and classifies `UNKNOWN`/missing entries via multilingual corporate legal forms, university markers, government/research institute terms, and personal-name regex rules. All organizational `UNKNOWN` entries are kept; individual inventors are excluded.
2. **Deterministic Name Cleaning (`clean`):** Normalizes Unicode/diacritics, extracts Latin tails from mixed CJK strings, strips `C/O` care-of clauses and incorporation boilerplate, standardizes recurring institutional tokens, drops filler prepositions/articles, and removes leading/trailing legal entity forms.

In [13]:
# --- Multilingual Organizational & Personal Name Patterns (EPO / PATSTAT Sector Imputation) ---
LEGAL_SEC = [
    "INC", "LLC", "LLLP", "LL", "LTD", "LIMITED", "CORP", "CORPORATION", "CO", "COMPANY", "COPMANY",
    "GMBH", "AG", "SE", "SA", "SAS", "SARL", "SRL", "SPA", "SL", "BV", "NV", "NA", "OY", "AB", "AS",
    "ASA", "PTY", "PLC", "KK", "LLP", "LP", "PTE", "KG", "OOO", "SP Z OO", "SPOLKA", "JOINT.STOCK",
    "ANONIM SIRKETI", "OBSHCHESTVO"
]
LEGAL_CJK_SEC = ["有限公司", "株式会社", "股份", "公司", "会社", "주식회사", "インコーポレイテッド", "アクチボラグ"]

UNIV_SEC = [
    "UNIV", "COLLEGE", "POLYTECHN", "POLITECNIC", "ECOLE", "HOCHSCHULE", "SCHOOL",
    "INSTITUTE OF TECHNOLOGY", "GIJUKU", "TRUSTEES", "EDUCATIONAL FUND",
    "大学", "大學", "学院", "カレッジ", "義塾", r"\bETH\b", r"\bEPFL\b", r"\bUESTC\b"
]

GOV_SEC = [
    "INSTIT", "INSTYTUT", "INSTUTUC", "ИНСТИТУТ", "ACADEM", "COMMISSARIAT", "CNRS", "LABORATORY",
    "CENTER", "CENTRE", "CENTRUM", "ZENTRUM", "FOUNDATION", "FONDAZIONE", "FUNDACIO", "STIFTUNG",
    "FRAUNHOFER", "FROUNHAV", "PLANCK", "GOVERNMENT", "MINISTRY", "MINISTERIUM", "BUNDES",
    "FEDERATSIYA", "AGENTSTVO", "UCHREZHDENIE", "SECRETARY", "AGENCY", "COUNCIL", "CONSEJO",
    "CONSIGLIO", r"\bNATIONAL\b", "IDRYMA", "ASSOCIATION", "HOSPITAL", "ARMY", "AIR FORCE", "NAVY",
    "RIKEN", r"\bTNO\b", "NASA", "研究", "中心", "委员会", "科学院", "センター"
]

FIRM_SEC = [
    "TECHNOLOG", "QUANT", "COMPUT", "SYSTEM", "SOLUTION", r"\bLABS?\b", "SOFTWARE", "NETWORK",
    "ELECTRONIC", "SEMICONDUCT", "PHOTONIC", "DEVICES", "INDUSTR", "HOLDING", "GROUP", "CAPITAL",
    "VENTURES", "ENTERPRISE", "INTERNATIONAL", "SCIENCE", "DIGITAL", "DATA", "INTELLIGEN", "ENERGY",
    "PHARMA", "SECUR", "MACHINES", "INSTRUMENTS", "ENGINEERING", "INNOVATION", "BLOCKCHAIN",
    "BANK", "INSURANCE", "PORTFOLIO", r"\bAI\b", "&", r"\+", "科技", "技术", "量子"
]

LEGAL_RE = re.compile(r"\b(" + "|".join(LEGAL_SEC) + r")\b|" + "|".join(LEGAL_CJK_SEC))
UNIV_RE, GOV_RE, FIRM_RE = (re.compile("|".join(words)) for words in (UNIV_SEC, GOV_SEC, FIRM_SEC))

WORD_PAT = r"[^\W\d_]+(?:['\-][^\W\d_]+)*"
PERSON_COMMA = re.compile(rf"^{WORD_PAT}( {WORD_PAT}){{0,3}} ?, ?{WORD_PAT}")
PERSON_WORDS = re.compile(rf"^{WORD_PAT}( {WORD_PAT}){{1,3}}$")


def _norm_for_sector(name: str) -> str:
    s = unicodedata.normalize("NFKC", str(name))
    s = "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c)).upper()
    s = " ".join(s.replace("?", "").replace(".", "").split())
    return s.replace("UNIVERSITY CORPORATION", "UNIVERSITY")


def impute_psn_sector(psn_name: str, psn_sector: str) -> str:
    '''Keep known PATSTAT sectors; replace UNKNOWN/NaN using EPO-style multilingual rules.'''
    s = str(psn_sector).strip().upper() if pd.notna(psn_sector) else ""
    if s and s != "UNKNOWN":
        if "UNIVERSITY" in s:             return "UNIVERSITY"
        if "GOV" in s or "HOSPITAL" in s: return "GOV NON-PROFIT"
        if "COMPANY" in s:                return "COMPANY"
        if "INDIVIDUAL" in s:             return "INDIVIDUAL"
        return s

    n = _norm_for_sector(psn_name)
    if LEGAL_RE.search(n): return "COMPANY"
    if UNIV_RE.search(n):  return "UNIVERSITY"
    if GOV_RE.search(n):   return "GOV NON-PROFIT"
    if FIRM_RE.search(n):  return "COMPANY"
    if PERSON_COMMA.match(n) or PERSON_WORDS.match(n) or "·" in n:
        return "INDIVIDUAL"
    if len(n.split()) == 1:
        return "COMPANY"
    return "UNCERTAIN"


# --- Deterministic String Normalization ---
FORMS = [
    "INC", "INCORPORATED", "LLC", "LLP", "LP", "LTD", "LIMITED", "CORP", "CORPORATION",
    "CO", "COMPANY", "PLC", "GMBH", "MBH", "AG", "KG", "SE", "SA", "SAS", "SARL", "SRL",
    "SPA", "SL", "BV", "NV", "OY", "OYJ", "AB", "AS", "ASA", "APS", "PTY", "PTE", "KK",
    "KABUSHIKI KAISHA", "SP Z OO", "OOO", "BHD", "SDN", "PVT", "LTDA", "LIMITEE", "LTEE",
    "AKTIENGESELLSCHAFT", "SOCIETE ANONYME", "SOCIEDAD ANONIMA", "SOCIETA PER AZIONI"
]
FORMS_RE = "|".join(sorted(FORMS, key=len, reverse=True))
FORMS_CJK = "股份有限公司|有限责任公司|有限責任公司|有限公司|株式会社|有限会社|合同会社|주식회사|公司"
WORDS = {
    "UNIVERSITY": "UNIV", "UNIVERSITAET": "UNIV", "UNIVERSITAT": "UNIV", "UNIVERSITE": "UNIV",
    "UNIVERSIDAD": "UNIV", "UNIVERSITA": "UNIV", "INSTITUTE": "INST", "INSTITUT": "INST",
    "INSTITUTO": "INST", "TECHNOLOGY": "TECH", "TECHNOLOGIES": "TECH", "TECHNOLOGICAL": "TECH",
    "LABORATORY": "LAB", "LABORATORIES": "LAB", "LABS": "LAB", "CENTRE": "CENTER",
    "SCIENCES": "SCIENCE", "大學": "大学"
}
FILLER = {"OF", "THE", "FOR", "AND", "DE", "DI", "DU", "DES", "ET", "UND"}


def clean(name):
    if pd.isna(name):
        return ""
    s = unicodedata.normalize("NFKD", unicodedata.normalize("NFKC", str(name)).upper())
    s = unicodedata.normalize("NFC", "".join(c for i, c in enumerate(s) if not (unicodedata.combining(c) and s[i - 1] < "ɐ")))
    s = re.sub(r"[▲▼?.'’]", "", s)
    latin = re.search(r"[^\x00-\x7f]([A-Z0-9][A-Z0-9 ,&\-/]+)$", s)
    if latin and len(re.sub(r"[^A-Z]", "", latin.group(1))) >= 3:
        s = latin.group(1)
    s = re.sub(r"^C/O\s+", "", s).strip()
    s = re.sub(r"(?:,|\s)\s*\bC/O\b.*$", "", s)
    s = re.sub(r"\(\s*A\s+[A-Z ]+\b(?:CORP|CORPORATION|COMPANY|LIMITED|LTD|INC)\b[^)]*\)", " ", s)
    s = s.replace("A/S", "AS").replace("&", " AND ")
    s = re.sub(rf"^({FORMS_CJK})|({FORMS_CJK})$", "", re.sub(r"[^\w ]|_", " ", s).strip())
    s = " ".join(WORDS.get(w, w) for w in s.split() if w not in FILLER) or s
    s = re.sub(rf"^(({FORMS_RE}) )+|( ({FORMS_RE}))+$", "", s).strip() or s
    return s


# --- Load & Preprocess Raw Data ---
raw_df = pd.read_csv(INPUT_CSV)
if SECTOR_COL not in raw_df.columns:
    raw_df[SECTOR_COL] = np.nan
raw_df["psn_sector_imputed"] = [
    impute_psn_sector(n, sec) for n, sec in zip(raw_df[NAME_COL], raw_df[SECTOR_COL])
]

# Known borderline overrides
raw_df.loc[raw_df[NAME_COL].isin([
    "DHITECH - DISTRETTO TECNOLOGICO H IGH TECH S.C.A R.L.",
    "PEARL COHEN ZEDEK LATZER BARATZ"
]), "psn_sector_imputed"] = "COMPANY"
raw_df.loc[raw_df[NAME_COL].str.contains("EHNSEHRM", na=False), "psn_sector_imputed"] = "GOV NON-PROFIT"

# Retain all organizational rows (including imputed UNKNOWN organizations); exclude individuals
df = raw_df[
    
    (raw_df[NAME_COL].notna()) &
    (raw_df[NAME_COL].str.strip() != "-NOT AVAILABLE-")
].copy()

df["clean"] = df[NAME_COL].apply(clean)
df = df[df["clean"].str.len() > 0].copy()
unique_names = sorted(df["clean"].unique())

print(f"Total raw rows: {len(raw_df):,} | Retained org rows: {len(df):,} | Unique cleaned names: {len(unique_names):,}")
print("\nImputed sector distribution across retained rows:")
print(df["psn_sector_imputed"].value_counts())

Total raw rows: 3,192 | Retained org rows: 3,192 | Unique cleaned names: 2,971

Imputed sector distribution across retained rows:
psn_sector_imputed
COMPANY           1730
INDIVIDUAL        1188
UNCERTAIN          216
GOV NON-PROFIT      55
UNIVERSITY           3
Name: count, dtype: int64


## 3. Multi-Signal Pairwise Scoring, Otsu Thresholding & Louvain Clustering

Encodes all unique normalized organizational names with `paraphrase-multilingual-MiniLM-L12-v2`, computes pairwise composite similarity (`0.34 * cosine + 0.33 * idf_jaccard + 0.33 * edit_ratio`) with a zero-token-overlap typo guard (`ed >= 0.80`), selects the bimodal threshold via **Otsu's method**, and partitions the similarity graph via **Louvain community detection**.

In [14]:
def pair_features_sector(names, emb, w=SCORE_WEIGHTS, floor=SCORE_FLOOR):
    n = len(names)
    wc, wj, we = w["cosine"], w["jaccard"], w["edit"]
    tokens = [set(s.split()) for s in names]
    lens = np.array([len(s) for s in names], dtype=np.float32)

    df_counts = Counter(t for toks in tokens for t in toks)
    idf = {t: np.log(n / c) for t, c in df_counts.items()}
    tok_idf_sums = np.array([sum(idf[t] for t in toks) for toks in tokens], dtype=np.float32)

    sim_matrix = emb @ emb.T
    min_cos = (floor - wj - we) / wc
    rows, cols = np.where(np.triu(sim_matrix >= min_cos, k=1))

    out = []
    for i, j in zip(rows, cols):
        cos = float(sim_matrix[i, j])
        ta, tb = tokens[i], tokens[j]
        inter = ta & tb
        if inter:
            inter_idf = sum(idf[t] for t in inter)
            union_idf = tok_idf_sums[i] + tok_idf_sums[j] - inter_idf
            jac = float(inter_idf / union_idf) if union_idf > 0 else 0.0
        else:
            jac = 0.0

        if wc * cos + wj * jac + we < floor:
            continue

        li, lj = lens[i], lens[j]
        max_ed = (2.0 * min(li, lj) / (li + lj)) if (li + lj) > 0 else 0.0
        if wc * cos + wj * jac + we * max_ed < floor:
            continue

        ed = rf_ratio(names[i], names[j]) / 100.0
        req_ed = max(0.0, (floor - wc * cos - wj * jac) / we)
        if jac == 0.0:
            req_ed = max(req_ed, 0.80)
        if req_ed > 1.0:
            continue

        score = wc * cos + wj * jac + we * ed
        if score >= floor:
            out.append((names[i], names[j], score))

    return pd.DataFrame(out, columns=["name_a", "name_b", "score"])


def find_threshold_otsu(scores, n_bins=100):
    counts, bin_edges = np.histogram(scores, bins=n_bins)
    centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    w1 = np.cumsum(counts)
    w2 = w1[-1] - w1
    valid = (w1 > 0) & (w2 > 0)
    w1, w2, centers_v = w1[valid], w2[valid], centers[valid]
    csum = np.cumsum(counts * centers)[valid]
    total_sum = np.sum(counts * centers)
    m1 = csum / w1
    m2 = (total_sum - csum) / w2
    between_var = w1 * w2 * (m1 - m2) ** 2
    return float(centers_v[np.argmax(between_var)])


model = SentenceTransformer(MODEL_NAME)
emb = model.encode(unique_names, normalize_embeddings=True, show_progress_bar=True).astype(np.float32)

scored = pair_features_sector(unique_names, emb)
raw_otsu = find_threshold_otsu(scored["score"].values) if len(scored) > 0 else MIN_OTSU_FLOOR
th_otsu = max(raw_otsu, MIN_OTSU_FLOOR)
print(f"Kept {len(scored):,} candidate edges | Unsupervised Otsu threshold: {th_otsu:.3f}")

G = nx.Graph()
G.add_nodes_from(unique_names)
edges = scored[scored["score"] >= th_otsu]
G.add_weighted_edges_from(zip(edges["name_a"], edges["name_b"], edges["score"]))
communities = nx_comm.louvain_communities(G, weight="weight", resolution=LOUVAIN_RESOLUTION_S1, seed=LOUVAIN_SEED)

if WEIGHT_COL and WEIGHT_COL in df.columns:
    name_weights = df.groupby("clean")[WEIGHT_COL].sum().to_dict()
else:
    name_weights = df["clean"].value_counts().to_dict()

sorted_comms = sorted(communities, key=lambda c: (-sum(name_weights.get(m, 0) for m in c), min(c)))

clean_to_canon_s1 = {}
clean_to_id_s1 = {}
for idx, members in enumerate(sorted_comms, start=1):
    harm_id = f"HARM_{idx:05d}"
    best_name = min(members, key=lambda n: (-name_weights.get(n, 0), len(n), n))
    for m in members:
        clean_to_canon_s1[m] = best_name
        clean_to_id_s1[m] = harm_id

df["harm_id"] = df["clean"].map(clean_to_id_s1)
df["harmonized_name"] = df["clean"].map(clean_to_canon_s1)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/93 [00:00<?, ?it/s]

Kept 27,156 candidate edges | Unsupervised Otsu threshold: 0.572


## 4. Stage 1 Export & Summary Diagnostics

Exports:
* **`outputs/harmonized_output_stage1.csv`:** Full organizational dataset enriched with `psn_sector_imputed`, `clean`, `harm_id`, and `harmonized_name`.
* **`outputs/harmonization_lookup_stage1.csv`:** Deduplicated lookup table used as the direct input to `stage2_3_specific_rectification.ipynb`.

In [15]:
from math import comb


def pairs_inside(group_sizes):
    "Number of pairs of names inside groups of the given sizes: n(n-1)/2 for each group."
    return sum(comb(int(n), 2) for n in group_sizes)


def precision_recall_f1(gold_groups, predicted_groups):
    "Pairwise precision, recall and F1 of a predicted grouping against the gold groups."
    df = pd.DataFrame({"gold": list(gold_groups), "pred": list(predicted_groups)})
    tp = pairs_inside(df.groupby(["gold", "pred"]).size())
    tp_fp = pairs_inside(df.groupby("pred").size())
    tp_fn = pairs_inside(df.groupby("gold").size())
    precision = tp / tp_fp if tp_fp else 1.0
    recall = tp / tp_fn
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"groups found": df["pred"].nunique(), "precision": precision, "recall": recall, "F1": f1}


# Stage 1 group id for every gold name; names dropped by Stage 1 (individuals, empty) stay alone
name_to_id = df.drop_duplicates(NAME_COL).set_index(NAME_COL)["harm_id"]
gold_eval = raw_df[["Group", "comp"]].copy()
gold_eval["group_id"] = gold_eval["comp"].map(name_to_id).fillna("DROPPED_" + gold_eval["comp"].astype(str))

gold_eval.to_csv("gold_stage1_eval.csv", index=False)

print(pd.Series(precision_recall_f1(gold_eval["Group"], gold_eval["group_id"])).round(3))

groups found    1000.000
precision          0.451
recall             0.198
F1                 0.275
dtype: float64
